<a href="https://colab.research.google.com/github/sagasucksatlife1/QuantProjects/blob/main/Momentum_Strategy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

In [85]:
import IPython
IPython.display.clear_output()

In [ ]:
tickers = [
    "RELIANCE.NS", "HDFCBANK.NS", "INFY.NS", "ITC.NS",
    "PIDILITIND.NS", "TRENT.NS", "ABB.NS",
        "IRCTC.NS", "CLEAN.NS", "SYNGENE.NS"]
portfolio_data = yf.download(tickers, start="2024-01-01")["Close"]
portfolio_data_5 = yf.download(tickers, period="7d",interval= "5m")["Close"]
portfolio_data_5 = portfolio_data_5.dropna(axis=1)
df = yf.download(tickers, period="7d", interval="5m")[["Open", "High", "Low", "Close", "Volume"]]

returns = portfolio_data_5.pct_change()
returns = returns.dropna()
returns.head()
returns.shape

/tmp/ipykernel_386/3187788044.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  portfolio_data = yf.download(tickers, start="2024-01-01")["Close"]
[*********************100%***********************]  10 of 10 completed
/tmp/ipykernel_386/3187788044.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  portfolio_data_5 = yf.download(tickers, period="7d",interval= "5m")["Close"]
[*********************100%***********************]  10 of 10 completed
/tmp/ipykernel_386/3187788044.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(tickers, period="7d", interval="5m")[["Open", "High", "Low", "Close", "Volume"]]
[*********************100%***********************]  10 of 10 completed


(474, 10)

In [ ]:
def autocorrelation(series, lag):
  return series.autocorr(lag=lag)

autocorr_lag1= returns.apply(lambda x: autocorrelation(x, 3))
autocorr_lag5= returns.apply(lambda x: autocorrelation(x, 5))
print("Lag 1d Autocorrelation:\n")
print(autocorr_lag1)
print("Lag 5d Autocorrelation:\n")
print(autocorr_lag5)


Lag 1d Autocorrelation:

Ticker
ABB.NS           0.019674
CLEAN.NS         0.015031
HDFCBANK.NS      0.066764
INFY.NS          0.100980
IRCTC.NS        -0.067217
ITC.NS          -0.010175
PIDILITIND.NS   -0.017174
RELIANCE.NS      0.151744
SYNGENE.NS       0.092570
TRENT.NS        -0.010252
dtype: float64
Lag 5d Autocorrelation:

Ticker
ABB.NS           0.008434
CLEAN.NS        -0.100580
HDFCBANK.NS     -0.077755
INFY.NS          0.022962
IRCTC.NS         0.035726
ITC.NS          -0.059960
PIDILITIND.NS   -0.009079
RELIANCE.NS     -0.061917
SYNGENE.NS       0.059305
TRENT.NS         0.041148
dtype: float64


In [ ]:
def momentum(series):
    if series.iloc[-1] > series.iloc[-2]:
        return True
    else:
        return False

momentum_series = returns.apply(momentum)
combined_1 = (momentum_series) | (autocorr_lag1 > 0)
combined_2 = (momentum_series) | (autocorr_lag5 > 0)
final = (combined_1) & (combined_2)
print("The stocks exibiting short and long term momentum are" ,final)


The stocks exibiting short and long term momentum are Ticker
ABB.NS            True
CLEAN.NS          True
HDFCBANK.NS      False
INFY.NS           True
IRCTC.NS         False
ITC.NS           False
PIDILITIND.NS    False
RELIANCE.NS       True
SYNGENE.NS        True
TRENT.NS          True
dtype: bool


In [ ]:
from scipy import stats

def hurst_exponent(ts):
  ts = np.array(ts)
  lags= range(2,50)
  tau = [np.std(ts[lag:] - ts[:-lag]) for lag in lags]
  poly = np.polyfit(np.log(lags),np.log(tau),1)
  return poly[0]

H1 = hurst_exponent(portfolio_data["ABB.NS"])
H2 = hurst_exponent(portfolio_data["INFY.NS"])
H3 = hurst_exponent(portfolio_data["SYNGENE.NS"])

print("Hurst exponent: ",H1,H2,H3)





Hurst exponent:  0.5366508804025137 0.5787005949324517 0.5198614325903012


In [ ]:
import pandas as pd

def rsi(series, period =14):
  series = pd.Series(series) # Ensure the input is a pandas Series
  delta = series.diff()
  gain = delta.where(delta>0,0)
  loss = -delta.where(delta<0,0)
  avg_gain = gain.rolling(window=period).mean()
  avg_loss = loss.rolling(window= period).mean()
  rs = avg_gain/avg_loss
  rsi = 100 - 100/(1+rs)
  return rsi

rsi_series = portfolio_data_5.apply(lambda x: rsi(x, period=21))
print(rsi_series)

Ticker                        ABB.NS   CLEAN.NS  HDFCBANK.NS    INFY.NS  \
Datetime                                                                  
2026-03-11 03:45:00+00:00        NaN        NaN          NaN        NaN   
2026-03-11 03:50:00+00:00        NaN        NaN          NaN        NaN   
2026-03-11 03:55:00+00:00        NaN        NaN          NaN        NaN   
2026-03-11 04:00:00+00:00        NaN        NaN          NaN        NaN   
2026-03-11 04:05:00+00:00        NaN        NaN          NaN        NaN   
...                              ...        ...          ...        ...   
2026-03-19 05:25:00+00:00  41.060419  39.233083    26.066689  16.959151   
2026-03-19 05:30:00+00:00  55.041322  47.585023    49.556432  30.961007   
2026-03-19 05:35:00+00:00  43.271222  33.560127    41.726619  28.125172   
2026-03-19 05:40:00+00:00  37.020316  36.609331    38.291592  31.153929   
2026-03-19 05:45:00+00:00  45.238095  41.944377    40.185502  33.333333   

Ticker                  

In [ ]:
def moving_avg_crossover(series,short_term,long_term):
  sma = series.rolling(window=short_term).mean()
  ema = series.ewm(span=long_term, adjust=False).mean()
  signal = sma > ema
  return signal


signal = moving_avg_crossover(portfolio_data_5, 10, 50)
latest_signal = signal.iloc[-1]
print(latest_signal)

Ticker
ABB.NS           False
CLEAN.NS         False
HDFCBANK.NS      False
INFY.NS          False
IRCTC.NS         False
ITC.NS           False
PIDILITIND.NS    False
RELIANCE.NS       True
SYNGENE.NS       False
TRENT.NS         False
Name: 2026-03-19 05:45:00+00:00, dtype: bool


In [ ]:
volume = yf.download(
    tickers,
    start="2026-01-01"
)["Volume"]
high_volume = volume > 0.7*volume.rolling(window=10).mean()


/tmp/ipykernel_386/3241031296.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  volume = yf.download(
[*********************100%***********************]  10 of 10 completed


In [ ]:
price = portfolio_data_5
sma = price.rolling(20).mean()
std = price.rolling(20).std()

upper = sma + 2 * std
lower = sma - 2 * std

bbwidth = upper - lower
min_bbwidth_10 = bbwidth.rolling(10).min()
squeeze = bbwidth <= 1.8 * min_bbwidth_10
print(squeeze)


Ticker                     ABB.NS  CLEAN.NS  HDFCBANK.NS  INFY.NS  IRCTC.NS  \
Datetime                                                                      
2026-03-11 03:45:00+00:00   False     False        False    False     False   
2026-03-11 03:50:00+00:00   False     False        False    False     False   
2026-03-11 03:55:00+00:00   False     False        False    False     False   
2026-03-11 04:00:00+00:00   False     False        False    False     False   
2026-03-11 04:05:00+00:00   False     False        False    False     False   
...                           ...       ...          ...      ...       ...   
2026-03-19 05:25:00+00:00    True      True         True     True      True   
2026-03-19 05:30:00+00:00    True      True         True     True      True   
2026-03-19 05:35:00+00:00    True      True         True     True      True   
2026-03-19 05:40:00+00:00    True      True         True     True      True   
2026-03-19 05:45:00+00:00    True      True         

In [ ]:
import pandas as pd

def rsi(series, period=14):
    series = pd.Series(series)
    delta = series.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.rolling(window=period).mean()
    avg_loss = loss.rolling(window=period).mean()
    rs = avg_gain / avg_loss
    rsi = 100 - 100 / (1 + rs)
    return rsi

def sma_indicator(series, period):
    series = pd.Series(series)
    return series.rolling(period).mean()

def std_indicator(series, period):
    series = pd.Series(series)
    return series.rolling(period).std()

def upper_bb(sma, std):
    return sma + 2 * std

def lower_bb(sma, std):
    return sma - 2 * std

def bb_width_indicator(upper, lower):
    return upper - lower

def min_bb_width(bbwidth_series, period):
    bbwidth_series = pd.Series(bbwidth_series)
    return bbwidth_series.rolling(period).min()


from backtesting import Strategy, Backtest

class MomentumStar(Strategy):

    def init(self):
        price = pd.Series(self.data.Close)
        volume = pd.Series(self.data.Volume)

        self.rsi = self.I(rsi, price, 14)
        self.rsi_avg = self.I(sma_indicator, self.rsi, 21)

        self.sma = self.I(sma_indicator, price, 20)
        self.std = self.I(std_indicator, price, 20)

        self.upper = self.I(upper_bb, self.sma, self.std)
        self.lower = self.I(lower_bb, self.sma, self.std)

        self.bbwidth = self.I(bb_width_indicator, self.upper, self.lower)
        self.min_bbwidth_10 = self.I(min_bb_width, self.bbwidth, 10)

        self.vol_ma = self.I(sma_indicator, volume, 10)

    def next(self):
        price = self.data.Close[-1]

        rsi = self.rsi[-1]
        rsi_avg = self.rsi_avg[-1]
        rsi_diff = rsi - rsi_avg

        squeeze = self.bbwidth[-1] <= 2.0 * self.min_bbwidth_10[-1]

        price_near_mid = (
            price >= 0.95 * self.sma[-1] and
            price <= 1.05 * self.sma[-1]
        )

        high_volume = self.data.Volume[-1] > 1.2 * self.vol_ma[-1]

        if rsi_diff > 0 and squeeze and price_near_mid and high_volume:
            if not self.position:
                self.buy()
        if self.position:
           entry = self.trades[-1].entry_price if self.trades else price
           if price < entry * 0.98:
            self.position.close()
           elif rsi_diff < 0:
            self.position.close()
           elif rsi > 70 and price > self.upper[-1]:
            self.position.close()

# FIX: slice df to single stock before passing to Backtest
df_single_stock = df.xs("RELIANCE.NS", axis=1, level=1).dropna()

bt = Backtest(df_single_stock, MomentumStar, cash=10000, commission=0.002)
stats = bt.run()

print(stats)
bt.plot()

Backtest.run:   0%|          | 0/441 [00:00<?, ?bar/s]

Start                     2026-03-11 03:45...
End                       2026-03-19 05:45...
Duration                      8 days 02:00:00
Exposure Time [%]                    32.42105
Equity Final [$]                   9359.08175
Equity Peak [$]                    10111.4398
Commissions [$]                      726.1204
Return [%]                           -6.40918
Buy & Hold Return [%]                 0.60832
Return (Ann.) [%]                   -90.18675
Volatility (Ann.) [%]                 1.27976
CAGR [%]                            -87.31797
Sharpe Ratio                        -70.47137
Sortino Ratio                        -4.37752
Calmar Ratio                         -12.1208
Alpha [%]                            -6.53131
Beta                                  0.20076
Max. Drawdown [%]                    -7.44066
Avg. Drawdown [%]                    -1.13798
Max. Drawdown Duration        7 days 00:15:00
Avg. Drawdown Duration        0 days 23:51:00
# Trades                          

/usr/local/lib/python3.12/dist-packages/bokeh/util/serialization.py:242: UserWarning: no explicit representation of timezones available for np.datetime64
  return convert(array.astype("datetime64[us]"))


GridPlot(id='p3438', ...)